In [13]:
# Блок 1: Импорт библиотек и настройка устройства (CPU/CUDA)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split

import string
import unicodedata
from io import open
import glob
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_device(device)
print(f"Используется устройство: {device}")

Используется устройство: cpu


In [14]:
# Блок 2: Определение алфавита и функция очистки текста
allowed_characters = string.ascii_letters + " .,;'" + "_"
n_letters = len(allowed_characters)

def unicodeToAscii(s):
    """Преобразует строку Unicode в ASCII (например, Ślusàrski -> Slusarski)"""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in allowed_characters
    )

# Тест функции
test_name = 'Ślusàrski'
print(f"Преобразование '{test_name}' -> '{unicodeToAscii(test_name)}'")

Преобразование 'Ślusàrski' -> 'Slusarski'


In [15]:
def letterToIndex(letter):
    """Возвращает индекс буквы в allowed_characters, для неизвестных - индекс '_'"""
    if letter not in allowed_characters:
        return allowed_characters.find("_")
    else:
        return allowed_characters.find(letter)

def lineToTensor(line):
    """Преобразует имя в тензор размера (len(line), 1, n_letters) с one-hot кодированием"""
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor

# Тест: one-hot для буквы 'a' и для имени 'Ahn'
print("One-hot для 'a':", lineToTensor('a').shape)
print("One-hot для 'Ahn':", lineToTensor('Ahn').shape)

One-hot для 'a': torch.Size([1, 1, 58])
One-hot для 'Ahn': torch.Size([3, 1, 58])


In [16]:
class NamesDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        labels_set = set()
        self.data = []
        self.data_tensors = []
        self.labels = []
        self.labels_tensors = []

        # Чтение всех .txt файлов в директории
        text_files = glob.glob(os.path.join(data_dir, '*.txt'))
        for filename in text_files:
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label)
            lines = open(filename, encoding='utf-8').read().strip().split('\n')
            for name in lines:
                ascii_name = unicodeToAscii(name)
                if ascii_name:  # пропускаем пустые строки
                    self.data.append(ascii_name)
                    self.data_tensors.append(lineToTensor(ascii_name))
                    self.labels.append(label)

        self.labels_uniq = list(labels_set)
        # Кэшируем тензоры меток
        for idx in range(len(self.labels)):
            temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
            self.labels_tensors.append(temp_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.labels_tensors[idx], self.data_tensors[idx], self.labels[idx], self.data[idx]

In [17]:
data_path = "data/names"

# Проверка существования папки
data_available = os.path.exists(data_path)

if data_available:
    alldata = NamesDataset(data_path)
    print(f"Загружено {len(alldata)} имён, языков: {len(alldata.labels_uniq)}")
    print(f"Пример: {alldata[0]}")
    
    # Разделение 85% / 15%
    train_set, test_set = random_split(alldata, [0.85, 0.15], 
                                       generator=torch.Generator(device=device).manual_seed(2024))
    print(f"Обучающих примеров: {len(train_set)}, тестовых: {len(test_set)}")
else:
    print(f"Папка {data_path} не найдена.")
    print("Скачайте данные с: https://download.pytorch.org/tutorial/data.zip")
    print("Распакуйте архив так, чтобы появилась папка 'data/names'")
    # Создаем заглушки для предотвращения ошибок
    alldata = None
    train_set = None
    test_set = None

Папка data/names не найдена.
Скачайте данные с: https://download.pytorch.org/tutorial/data.zip
Распакуйте архив так, чтобы появилась папка 'data/names'


In [18]:
# Архитектура RNN с проверкой наличия данных
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=False)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        rnn_out, hidden = self.rnn(line_tensor)
        output = self.h2o(hidden[-1])
        output = self.softmax(output)
        return output

if data_available and alldata is not None:
    n_hidden = 128
    rnn = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq))
    print(rnn)
    print(f"Модель создана: входной размер={n_letters}, скрытый размер={n_hidden}, выходной размер={len(alldata.labels_uniq)}")
else:
    print("Модель не создана: данные не загружены")
    rnn = None

Модель не создана: данные не загружены


In [19]:
def label_from_output(output, output_labels):
    """Преобразует выход сети в название языка"""
    top_n, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i], label_i

# Тест на одном примере (только если модель создана)
if rnn is not None and data_available:
    test_tensor = lineToTensor('Albert')
    rnn.eval()
    with torch.no_grad():
        output = rnn(test_tensor)
    pred_lang, pred_idx = label_from_output(output, alldata.labels_uniq)
    print(f"Для имени 'Albert' модель предсказала: {pred_lang}")
else:
    print("Пропуск теста: модель не создана")

Пропуск теста: модель не создана


In [20]:
def train(rnn, training_data, n_epoch=10, n_batch_size=64, 
          report_every=5, learning_rate=0.2, criterion=nn.NLLLoss()):
    
    if training_data is None or len(training_data) == 0:
        print("Ошибка: нет данных для обучения")
        return []
    
    current_loss = 0
    all_losses = []
    rnn.train()
    optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)
    
    print(f"Начало обучения на {len(training_data)} примерах...")
    start = time.time()
    
    for epoch in range(1, n_epoch + 1):
        # Создаём батчи
        indices = list(range(len(training_data)))
        random.shuffle(indices)
        batches = np.array_split(indices, max(1, len(indices) // n_batch_size))
        
        epoch_loss = 0
        for batch in batches:
            if len(batch) == 0:
                continue
                
            batch_loss = 0
            for idx in batch:
                label_tensor, text_tensor, _, _ = training_data[idx]
                output = rnn(text_tensor)
                loss = criterion(output, label_tensor)
                batch_loss += loss
            
            # Оптимизация для батча
            batch_loss.backward()
            nn.utils.clip_grad_norm_(rnn.parameters(), 3)
            optimizer.step()
            optimizer.zero_grad()
            
            current_loss += batch_loss.item() / len(batch)
            epoch_loss += batch_loss.item() / len(batch)
        
        avg_loss = epoch_loss / len(batches)
        all_losses.append(avg_loss)
        
        if epoch % report_every == 0:
            print(f"Эпоха {epoch}/{n_epoch} ({epoch/n_epoch:.0%}): средний loss = {avg_loss:.4f}")
        current_loss = 0
    
    end = time.time()
    print(f"Обучение завершено за {end-start:.2f} секунд")
    return all_losses

In [21]:
if data_available and rnn is not None and train_set is not None:
    # Параметры обучения (можно менять)
    all_losses = train(rnn, train_set, n_epoch=15, learning_rate=0.15, 
                       n_batch_size=64, report_every=5)
    
    # График ошибки
    if all_losses:
        plt.figure(figsize=(8, 5))
        plt.plot(all_losses)
        plt.title('Динамика loss при обучении')
        plt.xlabel('Эпоха')
        plt.ylabel('Средний loss')
        plt.grid(True)
        plt.show()
    else:
        print("Обучение не дало результатов")
else:
    print("Обучение пропущено: данные или модель не доступны")

Обучение пропущено: данные или модель не доступны


In [22]:
def evaluate(rnn, testing_data, classes):
    """Оценка модели и построение матрицы ошибок"""
    if testing_data is None or len(testing_data) == 0:
        print("Нет данных для оценки")
        return None
        
    confusion = torch.zeros(len(classes), len(classes))
    rnn.eval()
    
    with torch.no_grad():
        for i in range(len(testing_data)):
            label_tensor, text_tensor, label, _ = testing_data[i]
            output = rnn(text_tensor)
            _, guess_i = label_from_output(output, classes)
            label_i = classes.index(label)
            confusion[label_i][guess_i] += 1
    
    # Нормализация по строкам
    for i in range(len(classes)):
        row_sum = confusion[i].sum()
        if row_sum > 0:
            confusion[i] = confusion[i] / row_sum
    
    # Визуализация
    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    cax = ax.matshow(confusion.cpu().numpy(), cmap='Blues')
    plt.colorbar(cax)
    
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes, rotation=90, fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel('Предсказанный язык')
    ax.set_ylabel('Истинный язык')
    ax.set_title('Матрица ошибок (нормализованная)')
    plt.tight_layout()
    plt.show()
    
    return confusion

# Запуск оценки
if data_available and rnn is not None and test_set is not None:
    conf_matrix = evaluate(rnn, test_set, alldata.labels_uniq)
    if conf_matrix is not None:
        print("Оценка завершена.")
else:
    print("Оценка пропущена: данные или модель не доступны")

Оценка пропущена: данные или модель не доступны
